# 02 · AI gateway: identity, token limits, kill lists
`infra/mccp/policies/ai-gateway.xml` is mirrored by `mccp.policy_sim`. Order: validate-jwt → resolve agent/user → identity binding → global kill (503) → killed agents/users (423) → `llm-token-limit` per agent (TPM 429, monthly quota 403) → per-user monthly quota (403) → `llm-emit-token-metric` → managed identity to Foundry.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mccp").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

RUN_LIVE = False  # True only after deploying infra/mccp and filling .env (see docs/mccp/PLAN.md)

from mccp.config import Settings
from mccp.report import Console
settings = Settings.from_env(mode="azure" if RUN_LIVE else "local")
c = Console(color=False)
print("mode:", settings.mode, "(SIMULATED)" if settings.is_local else "(LIVE)")

In [ ]:
from mccp.demo import build_local_runtime
rt = build_local_runtime(settings)
sim = rt.simulator
print(sim.config)

In [ ]:
r = rt.call("contoso-claims-triage", "maria.gomez@contoso-health.example", "Triage claim CLM-0001")
print(r.status, r.outcome, "remaining TPM:", r.remaining_tokens, "user quota left:", r.user_remaining_quota)

## Burst limit (429) and monthly quota (403)

In [ ]:
prompt = "Re-code encounter ENC-7781. " * 900
for i in range(8):
    r = rt.call("contoso-rev-cycle-coder", "j.chen@contoso-health.example", prompt, max_tokens=1024)
    print(i + 1, r.status, r.error, r.retry_after)
    if r.status == 429:
        sim.clock.advance(60)
    elif r.status != 200:
        break

## XML ↔ simulator mapping

In [ ]:
from mccp.policy_sim import XML_ELEMENT_MAP
xml = (ROOT / "infra/mccp/policies/ai-gateway.xml").read_text()
for tag, fn in XML_ELEMENT_MAP:
    print(f"{'ok ' if tag in xml else 'MISSING'} {tag:<52} -> {fn}")

## Live call (RUN_LIVE)

In [ ]:
if RUN_LIVE:
    from mccp.gateway import GatewayClient
    from mccp.okta import OktaClient
    token = OktaClient(settings).client_credentials_token()["access_token"]
    agent = next(a for a in rt.agents if a.agent_id == "contoso-claims-triage")
    print(GatewayClient(settings).chat(agent, token, "Hello from Contoso Health"))
else:
    print("RUN_LIVE=False: skipped live APIM call")